# Entraîner un générateur texte-vers-vidéo avec Weli

Ce notebook entraîne `text_to_video_mlp` par rétropropagation sur de **vrais clips vidéo fournis localement et associés à leurs légendes**. Il apprend à reconstruire quatre images échantillonnées par clip à partir de sa légende. Il ne fabrique pas un jeu de données synthétique et n'utilise pas de modèle pré-entraîné.

**Données requises :** placez les images extraites de vrais clips sous `Data/text_to_video/frames/<clip_id>/` et créez `Data/text_to_video/captions.csv` avec les colonnes `clip_id,caption` (une ligne par clip). Exemple : `clip_001,"un chat marche dans l'herbe"`. Chaque dossier de clip doit contenir ses images dans l'ordre, nommées par exemple `0001.jpg`, `0002.jpg`, etc. Les clips ne sont pas présents dans ce dépôt : la cellule de chargement s'arrêtera avec une erreur explicite tant que les fichiers ne seront pas ajoutés.

Cette architecture expérimentale prédit une courte séquence à basse résolution, sans modéliser explicitement la continuité temporelle. Elle sert à tester le pipeline d'entraînement, pas à générer des vidéos réalistes.

In [2]:
from pathlib import Path
import csv
import hashlib
import sys

import matplotlib.pyplot as plt
import numpy as np
from PIL import Image
from sklearn.model_selection import train_test_split

ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents) if (p / 'pyproject.toml').exists()), None)
if ROOT is None:
    raise RuntimeError('Ouvrez ce notebook depuis le dépôt Weli afin de trouver pyproject.toml.')
sys.path.insert(0, str(ROOT))

from weli.losses import MSE
from weli.models import text_to_video_mlp
from weli.models.save_load import save_model
from weli.optimizers import Adam

DATA_DIR = ROOT / 'Data' / 'text_to_video'
OUTPUT_DIR = ROOT / 'training' / 'generated'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
SEED = 42
np.random.seed(SEED)

FRAME_COUNT = 4
FRAME_HEIGHT = 16
FRAME_WIDTH = 16
CHANNELS = 3
TEXT_LENGTH = 16
EMBEDDING_DIM = 32

## 1. Charger les légendes et les images des clips

Les images sont uniformément échantillonnées dans chaque clip, converties en RGB, réduites à `16 × 16` et normalisées dans `[-1, 1]`. Au moins deux clips valides sont nécessaires pour séparer entraînement et validation.

In [ ]:
captions_file = DATA_DIR / 'captions.csv'
frames_dir = DATA_DIR / 'frames'
if not captions_file.is_file() or not frames_dir.is_dir():
    raise FileNotFoundError(
        f'Données vidéo introuvables. Créez {captions_file} avec les colonnes '
        "'clip_id,caption' et ajoutez les images réelles dans "
        f'{frames_dir}/<clip_id>/*.jpg (ou .png).'
    )

with captions_file.open(encoding='utf-8-sig', newline='') as file:
    rows = list(csv.DictReader(file))
if not rows or not {'clip_id', 'caption'}.issubset(rows[0]):
    raise ValueError("captions.csv doit contenir des colonnes 'clip_id' et 'caption'.")

extensions = {'.jpg', '.jpeg', '.png', '.bmp'}
captions, clips = [], []
for row in rows:
    clip_id, caption = row['clip_id'].strip(), row['caption'].strip()
    if not clip_id or not caption:
        continue
    clip_path = frames_dir / clip_id
    frame_paths = sorted(p for p in clip_path.iterdir() if p.is_file() and p.suffix.lower() in extensions) if clip_path.is_dir() else []
    if not frame_paths:
        continue
    selected = np.linspace(0, len(frame_paths) - 1, FRAME_COUNT).round().astype(int)
    frame_arrays = []
    for frame_index in selected:
        with Image.open(frame_paths[frame_index]) as image:
            image = image.convert('RGB').resize((FRAME_WIDTH, FRAME_HEIGHT), Image.Resampling.BILINEAR)
            frame_arrays.append(np.asarray(image, dtype=np.float32) / 127.5 - 1.0)
    clips.append(np.stack(frame_arrays))
    captions.append(caption)

if len(clips) < 2:
    raise ValueError(f'Seulement {len(clips)} clip(s) avec images et légendes valides ; il en faut au moins 2.')

videos = np.stack(clips).astype(np.float32)
targets = videos.reshape(len(videos), -1)
print(f'{len(captions)} clips réels chargés ; chaque cible contient {FRAME_COUNT} images de {FRAME_HEIGHT} × {FRAME_WIDTH}.')
print(f'Entrée vidéo aplatie : {targets.shape}')

ValueError: captions.csv doit contenir des colonnes 'clip_id' et 'caption'.

## 2. Vectoriser les légendes et créer les ensembles

Ce vectoriseur pédagogique attribue un vecteur déterministe à chaque mot. Pour une expérience de production, remplacez-le par un encodeur de texte appris sur votre corpus.

In [ ]:
def word_vector(word):
    seed = int.from_bytes(hashlib.blake2b(word.encode('utf-8'), digest_size=8).digest(), 'little')
    return np.random.default_rng(seed).normal(0.0, 0.25, EMBEDDING_DIM).astype(np.float32)

def encode_caption(caption):
    encoded = np.zeros((TEXT_LENGTH, EMBEDDING_DIM), dtype=np.float32)
    for index, word in enumerate(caption.lower().split()[:TEXT_LENGTH]):
        encoded[index] = word_vector(word)
    return encoded

text_features = np.stack([encode_caption(caption) for caption in captions])
indices = np.arange(len(captions))
train_indices, val_indices = train_test_split(indices, test_size=0.2, random_state=SEED)
x_train, x_val = text_features[train_indices], text_features[val_indices]
y_train, y_val = targets[train_indices], targets[val_indices]
print(f'{len(train_indices)} clips entraînement, {len(val_indices)} validation')

## 3. Entraîner sur les clips réels

L'entraînement optimise la MSE des pixels des images échantillonnées. Avec peu de clips, le modèle risque de mémoriser les exemples ; utilisez une séparation par source vidéo et davantage de données pour une évaluation sérieuse.

In [ ]:
model = text_to_video_mlp(
    text_shape=(TEXT_LENGTH, EMBEDDING_DIM),
    num_frames=FRAME_COUNT,
    frame_shape=(FRAME_HEIGHT, FRAME_WIDTH, CHANNELS),
    hidden_units=64,
)
history = model.fit(
    x_train, y_train,
    epochs=40,
    batch_size=min(8, len(x_train)),
    validation_data=(x_val, y_val),
    loss_fn=MSE(),
    optimizer=Adam(lr=0.001),
    verbose=1,
)

plt.plot(history['train_loss'], label='entraînement')
plt.plot(history['val_loss'], label='validation')
plt.xlabel('Époque')
plt.ylabel('MSE des pixels')
plt.legend()
plt.grid(True)
plt.show()

## 4. Prévisualiser une séquence et sauvegarder

La séquence affichée correspond à une légende de validation. Chaque image est une prédiction du réseau, et non une copie automatique de la vidéo source.

In [ ]:
sample = int(val_indices[0])
generated = model.predict(text_features[sample:sample + 1])
generated = generated.reshape(FRAME_COUNT, FRAME_HEIGHT, FRAME_WIDTH, CHANNELS)
real = videos[sample]

fig, axes = plt.subplots(2, FRAME_COUNT, figsize=(10, 5))
for frame_index in range(FRAME_COUNT):
    axes[0, frame_index].imshow(np.clip((real[frame_index] + 1) / 2, 0, 1))
    axes[1, frame_index].imshow(np.clip((generated[frame_index] + 1) / 2, 0, 1))
    axes[0, frame_index].set_title(f'Image {frame_index + 1}')
    axes[0, frame_index].axis('off')
    axes[1, frame_index].axis('off')
axes[0, 0].set_ylabel('Réel')
axes[1, 0].set_ylabel('Généré')
plt.suptitle(captions[sample])
plt.tight_layout()
plt.show()

save_model(model, str(OUTPUT_DIR / 'text-to-video.weli'))
np.save(OUTPUT_DIR / 'text-to-video-sample.npy', generated)
print(f'Modèle et séquence sauvegardés dans {OUTPUT_DIR}')